In [ ]:
# ============================================================
# PHASE 4 STABLE END-TO-END
# Light TimeAware Transformer Encoder + GELU
# NaN-SAFE + DRIVE DISCONNECT-SAFE VERSION
# ============================================================

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, gc, json, time, pickle, random, shutil, warnings
from glob import glob

import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ============================================================
# 1. REPRODUCIBILITY
# ============================================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# ============================================================
# 2. CONFIG
# ============================================================

PHASE3_DRIVE_DIR = "/content/drive/MyDrive/Instacart/phase3_outputs_final"
PHASE3_LOCAL_DIR = "/content/phase3_outputs_final"

PHASE4_DIR = "/content/drive/MyDrive/Instacart/phase4_outputs_transformer_encoder_gelu_nan_safe_v2"
os.makedirs(PHASE4_DIR, exist_ok=True)

NUM_EPOCHS = 5

LEARNING_RATE = 1e-4
INNER_BATCH_SIZE = 128
LABEL_SMOOTHING = 0.05
LOGIT_CLAMP_VALUE = 20.0

WEIGHT_DECAY = 1e-5
DROPOUT = 0.2
VAL_SPLIT = 0.10
SEED = 42
PRINT_EVERY_FILES = 25

EMBED_DIM_PRODUCT = 64
EMBED_DIM_AISLE = 8
EMBED_DIM_DEPT = 4
EMBED_DIM_DOW = 4
EMBED_DIM_HOUR = 4
TIME_FEATURE_DIM = 4

MODEL_DIM = 96
NUM_HEADS = 4
NUM_TRANSFORMER_LAYERS = 1
FFN_DIM = 192
DENSE_DIM = 64
RECENCY_BETA_INIT = 0.10

MODEL_VARIANTS = [
    {
        "name": "Light_TimeAwareTransformerEncoder_GELU_NaNSafe_V2",
        "activation": "gelu"
    }
]

FORCE_CONTINUE_TRAINING = True
FORCE_RECOPY_PHASE3_LOCAL = False

# ============================================================
# 3. HELPERS
# ============================================================

def save_pickle(obj, path):
    with open(path, "wb") as f:
        pickle.dump(obj, f)

def load_pickle(path):
    with open(path, "rb") as f:
        return pickle.load(f)

def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

def ensure_dir(path):
    os.makedirs(path, exist_ok=True)

def print_line():
    print("=" * 80)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def clean_logits(logits):
    logits = torch.nan_to_num(
        logits,
        nan=0.0,
        posinf=LOGIT_CLAMP_VALUE,
        neginf=-LOGIT_CLAMP_VALUE
    )
    logits = torch.clamp(
        logits,
        min=-LOGIT_CLAMP_VALUE,
        max=LOGIT_CLAMP_VALUE
    )
    return logits

# ============================================================
# 4. DRIVE DISCONNECT-SAFE COPY PHASE 3 DATA TO LOCAL
# ============================================================
# ============================================================
# 4. DRIVE DISCONNECT-TOLERANT COPY PHASE 3 DATA TO LOCAL
# Copies only what is needed and does NOT stop on one failed file
# ============================================================

def copy_phase3_to_local(src_dir, dst_dir, force_recopy=False):
    print_line()
    print("Preparing local Phase 3 working directory...")

    if not os.path.exists(src_dir):
        raise FileNotFoundError(
            f"Drive source folder not found or Drive disconnected: {src_dir}. "
            "Reconnect Drive and rerun."
        )

    if os.path.exists(dst_dir) and force_recopy:
        print("Removing existing local copy...")
        shutil.rmtree(dst_dir)

    os.makedirs(dst_dir, exist_ok=True)

    # Copy metadata first
    metadata_src = os.path.join(src_dir, "phase3_metadata.pkl")
    metadata_dst = os.path.join(dst_dir, "phase3_metadata.pkl")

    if not os.path.exists(metadata_dst) or os.path.getsize(metadata_dst) == 0:
        shutil.copy2(metadata_src, metadata_dst)

    metadata = load_pickle(metadata_dst)
    run_id = metadata["RUN_ID"]

    print("Using RUN_ID:", run_id)

    # Copy only required batch files, not every Phase 3 file
    files = []
    files += sorted(glob(os.path.join(src_dir, f"train_batch_*_{run_id}.pkl")))
    files += sorted(glob(os.path.join(src_dir, f"test_batch_*_{run_id}.pkl")))

    print("Required Phase 3 batch files:", len(files))

    copied = 0
    skipped = 0
    failed = []

    for i, src_file in enumerate(files, start=1):
        dst_file = os.path.join(dst_dir, os.path.basename(src_file))
        tmp_file = dst_file + ".tmp"

        if os.path.exists(dst_file) and os.path.getsize(dst_file) > 0:
            skipped += 1
            continue

        try:
            if os.path.exists(tmp_file):
                os.remove(tmp_file)

            shutil.copy2(src_file, tmp_file)
            os.replace(tmp_file, dst_file)
            copied += 1

        except Exception as e:
            failed.append((src_file, str(e)))

            if os.path.exists(tmp_file):
                try:
                    os.remove(tmp_file)
                except:
                    pass

            print(f"Warning: failed to copy file {i}/{len(files)}")
            print("File:", os.path.basename(src_file))
            print("Error:", e)
            print("Continuing with remaining files...")

        if i % 100 == 0 or i == len(files):
            print(f"Copy progress: {i}/{len(files)} required files checked")

    print_line()
    print("Local Phase 3 folder ready:", dst_dir)
    print("Copied files :", copied)
    print("Skipped files:", skipped)
    print("Failed files :", len(failed))

    if len(failed) > 0:
        print("Some files failed because Drive disconnected or timed out.")
        print("Training will continue if enough valid train/test batches exist.")
        print("You can rerun later with FORCE_RECOPY_PHASE3_LOCAL=False to fill missing files.")

copy_phase3_to_local(
    PHASE3_DRIVE_DIR,
    PHASE3_LOCAL_DIR,
    force_recopy=FORCE_RECOPY_PHASE3_LOCAL
)

PHASE3_DIR = PHASE3_LOCAL_DIR

# ============================================================
# 5. LOAD METADATA + FILES
# ============================================================

metadata_candidates = glob(os.path.join(PHASE3_DIR, "phase3_metadata*.pkl"))

if len(metadata_candidates) == 0:
    raise FileNotFoundError(f"No Phase 3 metadata found inside {PHASE3_DIR}")

stable_metadata = os.path.join(PHASE3_DIR, "phase3_metadata.pkl")

if os.path.exists(stable_metadata):
    metadata_path = stable_metadata
else:
    metadata_path = max(metadata_candidates, key=os.path.getmtime)

metadata = load_pickle(metadata_path)

print_line()
print("Loaded metadata from:", metadata_path)

for k, v in metadata.items():
    if isinstance(v, (int, float, str)):
        print(f"{k}: {v}")

RUN_ID = metadata["RUN_ID"]

print_line()
print("Using RUN_ID:", RUN_ID)

raw_train_batch_files = sorted(glob(os.path.join(PHASE3_DIR, f"train_batch_*_{RUN_ID}.pkl")))
raw_test_batch_files  = sorted(glob(os.path.join(PHASE3_DIR, f"test_batch_*_{RUN_ID}.pkl")))

print_line()
print("Raw train batch files:", len(raw_train_batch_files))
print("Raw test batch files :", len(raw_test_batch_files))

# ============================================================
# 6. VALIDATE BATCH FILES
# ============================================================

REQUIRED_KEYS = {"Xp", "Xa", "Xd", "Xdow", "Xhr", "Xdays", "y"}

def validate_batch_files(file_list, label="train"):
    valid_files = []
    invalid_files = []

    for i, fpath in enumerate(file_list, start=1):
        try:
            obj = load_pickle(fpath)

            if isinstance(obj, dict) and REQUIRED_KEYS.issubset(set(obj.keys())):
                valid_files.append(fpath)
            else:
                invalid_files.append((fpath, str(type(obj))))

            del obj
            gc.collect()

        except Exception as e:
            invalid_files.append((fpath, str(e)))

        if i % 100 == 0 or i == len(file_list):
            print(f"{label}: checked {i}/{len(file_list)} files")

    return valid_files, invalid_files

train_batch_files, invalid_train_files = validate_batch_files(raw_train_batch_files, label="train")
test_batch_files, invalid_test_files = validate_batch_files(raw_test_batch_files, label="test")

print_line()
print("Valid train batch files:", len(train_batch_files))
print("Valid test batch files :", len(test_batch_files))
print("Invalid train files    :", len(invalid_train_files))
print("Invalid test files     :", len(invalid_test_files))

if len(train_batch_files) == 0 or len(test_batch_files) == 0:
    raise FileNotFoundError("No valid train/test batch files found.")

# ============================================================
# 7. TRAIN / VALIDATION SPLIT
# ============================================================

set_seed(SEED)

all_train_files = train_batch_files.copy()
random.shuffle(all_train_files)

val_count = max(1, int(len(all_train_files) * VAL_SPLIT))
val_files = all_train_files[:val_count]
train_files = all_train_files[val_count:]

print_line()
print("Train files:", len(train_files))
print("Val files  :", len(val_files))
print("Test files :", len(test_batch_files))

save_json(
    {
        "seed": SEED,
        "val_split": VAL_SPLIT,
        "num_train_files": len(train_files),
        "num_val_files": len(val_files),
        "num_test_files": len(test_batch_files)
    },
    os.path.join(PHASE4_DIR, "split_info.json")
)

sample_batch = load_pickle(train_files[0])

print_line()
print("Sample batch keys:", sample_batch.keys())

for k, v in sample_batch.items():
    arr = np.array(v)
    print(f"{k}: shape={arr.shape}, dtype={arr.dtype}")

MAX_LEN = np.array(sample_batch["Xp"]).shape[1]

PRODUCT_VOCAB_SIZE = 25001
AISLE_VOCAB_SIZE   = 135
DEPT_VOCAB_SIZE    = 22
SAFE_MAX_DOW       = 7
SAFE_MAX_HOUR      = 24

print_line()
print("MAX_LEN:", MAX_LEN)
print("PRODUCT_VOCAB_SIZE:", PRODUCT_VOCAB_SIZE)
print("AISLE_VOCAB_SIZE  :", AISLE_VOCAB_SIZE)
print("DEPT_VOCAB_SIZE   :", DEPT_VOCAB_SIZE)

# ============================================================
# 8. DATA LOADER
# ============================================================

def load_batch_tensors(batch_path, device=DEVICE):
    batch = load_pickle(batch_path)

    Xp    = torch.tensor(batch["Xp"],    dtype=torch.long,  device=device)
    Xa    = torch.tensor(batch["Xa"],    dtype=torch.long,  device=device)
    Xd    = torch.tensor(batch["Xd"],    dtype=torch.long,  device=device)
    Xdow  = torch.tensor(batch["Xdow"],  dtype=torch.long,  device=device)
    Xhr   = torch.tensor(batch["Xhr"],   dtype=torch.long,  device=device)
    Xdays = torch.tensor(batch["Xdays"], dtype=torch.float, device=device)
    y     = torch.tensor(batch["y"],     dtype=torch.long,  device=device)

    return {
        "Xp": Xp,
        "Xa": Xa,
        "Xd": Xd,
        "Xdow": Xdow,
        "Xhr": Xhr,
        "Xdays": Xdays,
        "y": y
    }

def iterate_inner_batches(batch_tensors, inner_batch_size=128):
    n = batch_tensors["y"].shape[0]

    for start in range(0, n, inner_batch_size):
        end = min(start + inner_batch_size, n)

        yield {
            "Xp": batch_tensors["Xp"][start:end],
            "Xa": batch_tensors["Xa"][start:end],
            "Xd": batch_tensors["Xd"][start:end],
            "Xdow": batch_tensors["Xdow"][start:end],
            "Xhr": batch_tensors["Xhr"][start:end],
            "Xdays": batch_tensors["Xdays"][start:end],
            "y": batch_tensors["y"][start:end]
        }

# ============================================================
# 9. METRICS
# ============================================================

def compute_classification_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    rec = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)

    return {
        "accuracy": float(acc),
        "precision_weighted": float(prec),
        "recall_weighted": float(rec),
        "f1_weighted": float(f1)
    }

# ============================================================
# 10. TRANSFORMER MODEL
# ============================================================

class LightTimeAwareTransformerEncoder(nn.Module):
    def __init__(
        self,
        product_vocab_size,
        aisle_vocab_size,
        dept_vocab_size,
        max_dow,
        max_hour,
        max_len,
        embed_dim_product=64,
        embed_dim_aisle=8,
        embed_dim_dept=4,
        embed_dim_dow=4,
        embed_dim_hour=4,
        time_feature_dim=4,
        model_dim=96,
        num_heads=4,
        num_layers=1,
        ffn_dim=192,
        dense_dim=64,
        dropout=0.2,
        recency_beta_init=0.10
    ):
        super().__init__()

        self.product_emb = nn.Embedding(product_vocab_size, embed_dim_product, padding_idx=0)
        self.aisle_emb   = nn.Embedding(aisle_vocab_size, embed_dim_aisle, padding_idx=0)
        self.dept_emb    = nn.Embedding(dept_vocab_size, embed_dim_dept, padding_idx=0)
        self.dow_emb     = nn.Embedding(max_dow, embed_dim_dow, padding_idx=0)
        self.hour_emb    = nn.Embedding(max_hour, embed_dim_hour, padding_idx=0)

        self.time_mlp = nn.Sequential(
            nn.Linear(1, time_feature_dim),
            nn.ReLU(),
            nn.Linear(time_feature_dim, time_feature_dim)
        )

        input_dim = (
            embed_dim_product +
            embed_dim_aisle +
            embed_dim_dept +
            embed_dim_dow +
            embed_dim_hour +
            time_feature_dim +
            1
        )

        self.input_projection = nn.Linear(input_dim, model_dim)
        self.position_emb = nn.Embedding(max_len, model_dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=model_dim,
            nhead=num_heads,
            dim_feedforward=ffn_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.recency_beta = nn.Parameter(torch.tensor(recency_beta_init, dtype=torch.float32))

        self.fc1 = nn.Linear(model_dim, dense_dim)
        self.activation = nn.GELU()
        self.dropout = nn.Dropout(dropout)
        self.fc_out = nn.Linear(dense_dim, product_vocab_size)

    def masked_mean_pooling(self, encoder_outputs, mask):
        mask = mask.unsqueeze(-1).float()
        summed = torch.sum(encoder_outputs * mask, dim=1)
        count = torch.clamp(mask.sum(dim=1), min=1.0)
        pooled = summed / count
        return pooled

    def forward(self, Xp, Xa, Xd, Xdow, Xhr, Xdays):
        batch_size, seq_len = Xp.shape

        p_emb  = self.product_emb(Xp)
        a_emb  = self.aisle_emb(Xa)
        d_emb  = self.dept_emb(Xd)
        dw_emb = self.dow_emb(Xdow)
        hr_emb = self.hour_emb(Xhr)

        Xdays = torch.nan_to_num(Xdays, nan=0.0, posinf=365.0, neginf=0.0)
        Xdays = torch.clamp(Xdays, min=0.0, max=365.0)

        xdays_log = torch.log1p(Xdays)
        denom = torch.clamp(xdays_log.max().detach(), min=1e-8)
        xdays_norm = xdays_log / denom

        gap_feature = xdays_norm.unsqueeze(-1)
        time_encoded = self.time_mlp(gap_feature)

        x = torch.cat(
            [p_emb, a_emb, d_emb, dw_emb, hr_emb, time_encoded, gap_feature],
            dim=-1
        )

        beta = torch.clamp(self.recency_beta, min=0.0, max=2.0)
        recency_weight = torch.exp(-beta * xdays_norm).unsqueeze(-1)

        x = x * recency_weight
        x = self.input_projection(x)

        positions = torch.arange(seq_len, device=Xp.device).unsqueeze(0).expand(batch_size, seq_len)
        x = x + self.position_emb(positions)

        valid_mask = Xp != 0
        padding_mask = Xp == 0

        encoder_outputs = self.transformer_encoder(
            x,
            src_key_padding_mask=padding_mask
        )

        encoder_outputs = torch.nan_to_num(
            encoder_outputs,
            nan=0.0,
            posinf=LOGIT_CLAMP_VALUE,
            neginf=-LOGIT_CLAMP_VALUE
        )

        context = self.masked_mean_pooling(encoder_outputs, valid_mask)

        h = self.fc1(context)
        h = self.activation(h)
        h = self.dropout(h)

        logits = self.fc_out(h)
        logits = clean_logits(logits)

        return logits, None

def build_model():
    model = LightTimeAwareTransformerEncoder(
        product_vocab_size=PRODUCT_VOCAB_SIZE,
        aisle_vocab_size=AISLE_VOCAB_SIZE,
        dept_vocab_size=DEPT_VOCAB_SIZE,
        max_dow=SAFE_MAX_DOW,
        max_hour=SAFE_MAX_HOUR,
        max_len=MAX_LEN,
        embed_dim_product=EMBED_DIM_PRODUCT,
        embed_dim_aisle=EMBED_DIM_AISLE,
        embed_dim_dept=EMBED_DIM_DEPT,
        embed_dim_dow=EMBED_DIM_DOW,
        embed_dim_hour=EMBED_DIM_HOUR,
        time_feature_dim=TIME_FEATURE_DIM,
        model_dim=MODEL_DIM,
        num_heads=NUM_HEADS,
        num_layers=NUM_TRANSFORMER_LAYERS,
        ffn_dim=FFN_DIM,
        dense_dim=DENSE_DIM,
        dropout=DROPOUT,
        recency_beta_init=RECENCY_BETA_INIT
    )

    return model.to(DEVICE)

# ============================================================
# 11. CHECKPOINT HELPERS
# ============================================================

def get_model_dir(model_name):
    model_dir = os.path.join(PHASE4_DIR, model_name)
    ensure_dir(model_dir)
    return model_dir

def get_paths(model_name):
    model_dir = get_model_dir(model_name)

    return {
        "model_dir": model_dir,
        "latest_ckpt": os.path.join(model_dir, "checkpoint_latest.pt"),
        "best_ckpt": os.path.join(model_dir, "checkpoint_best.pt"),
        "history_csv": os.path.join(model_dir, "training_history.csv"),
        "metrics_json": os.path.join(model_dir, "final_metrics.json"),
        "config_json": os.path.join(model_dir, "model_config.json"),
        "done_flag": os.path.join(model_dir, "TRAINING_DONE.flag")
    }

def save_checkpoint(path, epoch, model, optimizer, best_val_f1, history, variant):
    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_val_f1": best_val_f1,
        "history": history,
        "variant": variant
    }

    torch.save(checkpoint, path)

def load_checkpoint(path, model, optimizer=None):
    checkpoint = torch.load(path, map_location=DEVICE)

    model.load_state_dict(checkpoint["model_state_dict"])

    if optimizer is not None and "optimizer_state_dict" in checkpoint:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    epoch = checkpoint.get("epoch", 0)
    best_val_f1 = checkpoint.get("best_val_f1", -1.0)
    history = checkpoint.get("history", [])
    variant = checkpoint.get("variant", None)

    return model, optimizer, epoch, best_val_f1, history, variant

# ============================================================
# 12. TRAIN / EVAL
# ============================================================

def train_one_epoch(model, optimizer, batch_files, criterion, inner_batch_size=128):
    model.train()

    running_loss = 0.0
    total_samples = 0

    all_y_true = []
    all_y_pred = []

    skipped_nan_batches = 0

    for file_idx, batch_path in enumerate(batch_files):
        batch_tensors = load_batch_tensors(batch_path, device=DEVICE)

        for mini_batch in iterate_inner_batches(batch_tensors, inner_batch_size=inner_batch_size):
            optimizer.zero_grad()

            logits, _ = model(
                mini_batch["Xp"],
                mini_batch["Xa"],
                mini_batch["Xd"],
                mini_batch["Xdow"],
                mini_batch["Xhr"],
                mini_batch["Xdays"]
            )

            logits = clean_logits(logits)
            loss = criterion(logits, mini_batch["y"])

            if torch.isnan(loss) or torch.isinf(loss):
                skipped_nan_batches += 1
                del mini_batch, logits, loss
                continue

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            batch_size = mini_batch["y"].size(0)

            running_loss += loss.item() * batch_size
            total_samples += batch_size

            preds = torch.argmax(logits, dim=1)

            all_y_true.extend(mini_batch["y"].detach().cpu().numpy().tolist())
            all_y_pred.extend(preds.detach().cpu().numpy().tolist())

            del mini_batch, logits, loss, preds

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        del batch_tensors
        gc.collect()

        if (file_idx + 1) % PRINT_EVERY_FILES == 0 or (file_idx + 1) == len(batch_files):
            print(f"Train progress: {file_idx + 1}/{len(batch_files)} files processed")

    if total_samples == 0:
        return float("nan"), {
            "accuracy": 0.0,
            "precision_weighted": 0.0,
            "recall_weighted": 0.0,
            "f1_weighted": 0.0
        }

    epoch_loss = running_loss / total_samples
    metrics = compute_classification_metrics(all_y_true, all_y_pred)

    print("Skipped NaN/Inf train batches:", skipped_nan_batches)

    return epoch_loss, metrics

@torch.no_grad()
def evaluate_model(model, batch_files, criterion, inner_batch_size=128):
    model.eval()

    running_loss = 0.0
    total_samples = 0

    all_y_true = []
    all_y_pred = []

    skipped_nan_batches = 0

    for file_idx, batch_path in enumerate(batch_files):
        batch_tensors = load_batch_tensors(batch_path, device=DEVICE)

        for mini_batch in iterate_inner_batches(batch_tensors, inner_batch_size=inner_batch_size):
            logits, _ = model(
                mini_batch["Xp"],
                mini_batch["Xa"],
                mini_batch["Xd"],
                mini_batch["Xdow"],
                mini_batch["Xhr"],
                mini_batch["Xdays"]
            )

            logits = clean_logits(logits)
            loss = criterion(logits, mini_batch["y"])

            if torch.isnan(loss) or torch.isinf(loss):
                skipped_nan_batches += 1
                del mini_batch, logits, loss
                continue

            batch_size = mini_batch["y"].size(0)

            running_loss += loss.item() * batch_size
            total_samples += batch_size

            preds = torch.argmax(logits, dim=1)

            all_y_true.extend(mini_batch["y"].detach().cpu().numpy().tolist())
            all_y_pred.extend(preds.detach().cpu().numpy().tolist())

            del mini_batch, logits, loss, preds

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        del batch_tensors
        gc.collect()

        if (file_idx + 1) % PRINT_EVERY_FILES == 0 or (file_idx + 1) == len(batch_files):
            print(f"Eval progress: {file_idx + 1}/{len(batch_files)} files processed")

    if total_samples == 0:
        return float("nan"), {
            "accuracy": 0.0,
            "precision_weighted": 0.0,
            "recall_weighted": 0.0,
            "f1_weighted": 0.0
        }

    epoch_loss = running_loss / total_samples
    metrics = compute_classification_metrics(all_y_true, all_y_pred)

    print("Skipped NaN/Inf eval batches:", skipped_nan_batches)

    return epoch_loss, metrics

# ============================================================
# 13. TRAIN MODEL
# ============================================================

def train_model_variant(variant):
    model_name = variant["name"]

    print_line()
    print(f"STARTING MODEL: {model_name}")
    print_line()

    paths = get_paths(model_name)

    if FORCE_CONTINUE_TRAINING and os.path.exists(paths["done_flag"]):
        os.remove(paths["done_flag"])
        print(f"Removed old done flag for continuation: {model_name}")

    if os.path.exists(paths["done_flag"]):
        print(f"{model_name} already finished earlier. Skipping training.")
        return

    model = build_model()

    print(f"Trainable params: {count_parameters(model):,}")

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

    start_epoch = 1
    best_val_f1 = -1.0
    history = []

    save_json(
        {
            "model_name": model_name,
            "model_type": "TransformerEncoder",
            "activation": "GELU",
            "num_epochs": NUM_EPOCHS,
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "dropout": DROPOUT,
            "label_smoothing": LABEL_SMOOTHING,
            "logit_clamp_value": LOGIT_CLAMP_VALUE,
            "seed": SEED,
            "max_len": MAX_LEN,
            "product_vocab_size": PRODUCT_VOCAB_SIZE,
            "aisle_vocab_size": AISLE_VOCAB_SIZE,
            "dept_vocab_size": DEPT_VOCAB_SIZE,
            "model_dim": MODEL_DIM,
            "num_heads": NUM_HEADS,
            "num_transformer_layers": NUM_TRANSFORMER_LAYERS,
            "ffn_dim": FFN_DIM,
            "dense_dim": DENSE_DIM,
            "inner_batch_size": INNER_BATCH_SIZE
        },
        paths["config_json"]
    )

    if os.path.exists(paths["latest_ckpt"]):
        print(f"Resuming from checkpoint: {paths['latest_ckpt']}")

        model, optimizer, last_epoch, best_val_f1, history, _ = load_checkpoint(
            paths["latest_ckpt"],
            model,
            optimizer
        )

        start_epoch = last_epoch + 1

        for param_group in optimizer.param_groups:
            param_group["lr"] = LEARNING_RATE

        print(f"Resumed from epoch {last_epoch}. New LR set to {LEARNING_RATE}")

        if start_epoch > NUM_EPOCHS:
            print(f"{model_name} already reached target epochs. Marking done.")

            with open(paths["done_flag"], "w") as f:
                f.write("done")

            return

    for epoch in range(start_epoch, NUM_EPOCHS + 1):
        print_line()
        print(f"{model_name} | Epoch {epoch}/{NUM_EPOCHS}")

        start_time = time.time()

        train_loss, train_metrics = train_one_epoch(
            model,
            optimizer,
            train_files,
            criterion,
            inner_batch_size=INNER_BATCH_SIZE
        )

        val_loss, val_metrics = evaluate_model(
            model,
            val_files,
            criterion,
            inner_batch_size=INNER_BATCH_SIZE
        )

        elapsed = time.time() - start_time

        epoch_record = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "train_accuracy": train_metrics["accuracy"],
            "train_precision_weighted": train_metrics["precision_weighted"],
            "train_recall_weighted": train_metrics["recall_weighted"],
            "train_f1_weighted": train_metrics["f1_weighted"],
            "val_accuracy": val_metrics["accuracy"],
            "val_precision_weighted": val_metrics["precision_weighted"],
            "val_recall_weighted": val_metrics["recall_weighted"],
            "val_f1_weighted": val_metrics["f1_weighted"],
            "epoch_time_sec": elapsed
        }

        history.append(epoch_record)

        print(f"Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f}")
        print(f"Train Acc : {train_metrics['accuracy']:.6f} | Val Acc : {val_metrics['accuracy']:.6f}")
        print(f"Train F1  : {train_metrics['f1_weighted']:.6f} | Val F1  : {val_metrics['f1_weighted']:.6f}")
        print(f"Epoch time: {elapsed:.2f} sec")

        save_checkpoint(
            paths["latest_ckpt"],
            epoch,
            model,
            optimizer,
            best_val_f1,
            history,
            variant
        )

        if not np.isnan(val_loss) and val_metrics["f1_weighted"] > best_val_f1:
            best_val_f1 = val_metrics["f1_weighted"]

            save_checkpoint(
                paths["best_ckpt"],
                epoch,
                model,
                optimizer,
                best_val_f1,
                history,
                variant
            )

            print("Saved new BEST checkpoint.")
        else:
            print("Best checkpoint not updated.")

        pd.DataFrame(history).to_csv(paths["history_csv"], index=False)

    with open(paths["done_flag"], "w") as f:
        f.write("done")

    print_line()
    print(f"TRAINING COMPLETE: {model_name}")

# ============================================================
# 14. FINAL TEST EVALUATION
# ============================================================

def final_test_evaluation(variant):
    model_name = variant["name"]
    paths = get_paths(model_name)

    if not os.path.exists(paths["best_ckpt"]):
        print(f"No best checkpoint found for {model_name}. Skipping test evaluation.")
        return None

    model = build_model()
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

    model, _, best_epoch, best_val_f1, history, _ = load_checkpoint(
        paths["best_ckpt"],
        model,
        optimizer=None
    )

    test_loss, test_metrics = evaluate_model(
        model,
        test_batch_files,
        criterion,
        inner_batch_size=INNER_BATCH_SIZE
    )

    result = {
        "model_name": model_name,
        "best_epoch": best_epoch,
        "best_val_f1": best_val_f1,
        "test_loss": test_loss,
        "test_accuracy": test_metrics["accuracy"],
        "test_precision_weighted": test_metrics["precision_weighted"],
        "test_recall_weighted": test_metrics["recall_weighted"],
        "test_f1_weighted": test_metrics["f1_weighted"]
    }

    save_json(result, paths["metrics_json"])

    print_line()
    print(f"FINAL TEST RESULTS: {model_name}")

    for k, v in result.items():
        print(f"{k}: {v}")

    return result

# ============================================================
# 15. RUN TRAINING
# ============================================================

for variant in MODEL_VARIANTS:
    train_model_variant(variant)

# ============================================================
# 16. RUN FINAL EVALUATION
# ============================================================

all_results = []

for variant in MODEL_VARIANTS:
    result = final_test_evaluation(variant)

    if result is not None:
        all_results.append(result)

results_df = pd.DataFrame(all_results)

comparison_csv = os.path.join(PHASE4_DIR, "phase4_transformer_encoder_gelu_nan_safe_v2_comparison.csv")
comparison_json = os.path.join(PHASE4_DIR, "phase4_transformer_encoder_gelu_nan_safe_v2_comparison.json")

results_df.to_csv(comparison_csv, index=False)
save_json(all_results, comparison_json)

print_line()
print("PHASE 4 TRANSFORMER ENCODER + GELU NAN-SAFE V2 COMPLETE")
print_line()

if len(results_df) > 0:
    display(results_df.sort_values(by="test_f1_weighted", ascending=False))
else:
    print("No model results found.")

Mounted at /content/drive
PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
Using device: cuda
Preparing local Phase 3 working directory...
Using RUN_ID: 20260422_185902
Required Phase 3 batch files: 2057
Copy progress: 100/2057 required files checked
Copy progress: 200/2057 required files checked
Copy progress: 300/2057 required files checked
Copy progress: 400/2057 required files checked
Copy progress: 500/2057 required files checked
Copy progress: 600/2057 required files checked
Copy progress: 700/2057 required files checked
Copy progress: 800/2057 required files checked
Copy progress: 900/2057 required files checked
Copy progress: 1000/2057 required files checked
Copy progress: 1100/2057 required files checked
Copy progress: 1200/2057 required files checked
Copy progress: 1300/2057 required files checked
Copy progress: 1400/2057 required files checked
Copy progress: 1500/2057 required files checked
Copy progress: 1600/2057 required files checked
Copy progress: 1700/2

,model_name,best_epoch,best_val_f1,test_loss,test_accuracy,test_precision_weighted,test_recall_weighted,test_f1_weighted
0,Light_TimeAwareTransformerEncoder_GELU_NaNSafe_V2,5,0.01016,7.764294,0.035615,0.014083,0.035615,0.010113
